In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Engineered Clinical Feature Distribution Visualization (`plots/plot_engineered.ipynb`)

This notebook computes and visualizes the distribution of **10 Engineered Clinical Features** specified in `TODO.md`:
1. `is_dyspnea_total` (`triage_vital_o2 < 90`)
2. `is_dyspnea_moderate` (`triage_vital_o2 >= 90 & < 94`)
3. `is_bradypnea` (`triage_vital_rr < 10`)
4. `is_tachypnea` (`triage_vital_rr > 30`)
5. `is_hypotension` (`triage_vital_sbp <= 90`)
6. `is_hypertension` (`triage_vital_sbp > 220`)
7. `is_bradycardia_total` (`triage_vital_hr < 40`)
8. `is_bradycardia_moderate` (`triage_vital_hr >= 40 & < 60`)
9. `is_tachycardia_total` (`triage_vital_hr > 150`)
10. `is_tachycardia_moderate` (`triage_vital_hr > 100 & <= 150`)

### Optimizations Applied:
- **Vectorized Loop Summary**: Computes prevalence per column efficiently without heavy memory allocation, guaranteeing fast execution and zero kernel crashes.
- **Exported Artifacts**: PNG plots exported to `plots/image/` and summary tables exported to `plots/csv/`.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
library(jsonlite)
library(ggplot2)
library(dplyr)
library(tidyr)

config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}

config <- fromJSON(config_path)

cat("=== Configuration Loaded ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Dataset & Ensure Output Directories
# ---------------------------------------------------------
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}

cat("Loading dataset from:", data_file, "...\n")

data_env <- new.env()
load(data_file, envir = data_env)

df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]

raw_df <- get(data_obj_name, envir = data_env)

target_col   <- config$classes$target_col
target_classes <- as.character(config$classes$outputs)

# Output directories
img_dir <- "../plots/image"
if (!dir.exists(img_dir)) img_dir <- "image"
if (!dir.exists(img_dir)) dir.create(img_dir, recursive = TRUE)

csv_dir <- "../plots/csv"
if (!dir.exists(csv_dir)) csv_dir <- "csv"
if (!dir.exists(csv_dir)) dir.create(csv_dir, recursive = TRUE)

cat(sprintf("Loaded dataset: %d rows x %d cols\n", nrow(raw_df), ncol(raw_df)))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Compute 10 Engineered Features & Plot Prevalence
# ---------------------------------------------------------
cat("=== Computing 10 Engineered Clinical Features from TODO.md ===\n")

df_eng <- raw_df

# Calculate 10 Engineered Features specified in TODO.md
df_eng$is_dyspnea_total       <- ifelse(!is.na(df_eng$triage_vital_o2) & df_eng$triage_vital_o2 < 90, 1, 0)
df_eng$is_dyspnea_moderate    <- ifelse(!is.na(df_eng$triage_vital_o2) & df_eng$triage_vital_o2 >= 90 & df_eng$triage_vital_o2 < 94, 1, 0)
df_eng$is_bradypnea           <- ifelse(!is.na(df_eng$triage_vital_rr) & df_eng$triage_vital_rr < 10, 1, 0)
df_eng$is_tachypnea           <- ifelse(!is.na(df_eng$triage_vital_rr) & df_eng$triage_vital_rr > 30, 1, 0)
df_eng$is_hypotension         <- ifelse(!is.na(df_eng$triage_vital_sbp) & df_eng$triage_vital_sbp <= 90, 1, 0)
df_eng$is_hypertension        <- ifelse(!is.na(df_eng$triage_vital_sbp) & df_eng$triage_vital_sbp > 220, 1, 0)
df_eng$is_bradycardia_total   <- ifelse(!is.na(df_eng$triage_vital_hr) & df_eng$triage_vital_hr < 40, 1, 0)
df_eng$is_bradycardia_moderate<- ifelse(!is.na(df_eng$triage_vital_hr) & df_eng$triage_vital_hr >= 40 & df_eng$triage_vital_hr < 60, 1, 0)
df_eng$is_tachycardia_total   <- ifelse(!is.na(df_eng$triage_vital_hr) & df_eng$triage_vital_hr > 150, 1, 0)
df_eng$is_tachycardia_moderate<- ifelse(!is.na(df_eng$triage_vital_hr) & df_eng$triage_vital_hr > 100 & df_eng$triage_vital_hr <= 150, 1, 0)

eng_cols <- c(
  "is_dyspnea_total", "is_dyspnea_moderate", "is_bradypnea", "is_tachypnea",
  "is_hypotension", "is_hypertension", "is_bradycardia_total", "is_bradycardia_moderate",
  "is_tachycardia_total", "is_tachycardia_moderate"
)

df_eng[[target_col]] <- factor(df_eng[[target_col]], levels = target_classes)

# Fast column-wise prevalence calculation (Ultra lightweight - zero memory spike)
eng_list <- list()
for (col_name in eng_cols) {
  sub_summary <- df_eng %>%
    group_by(ESI = .data[[target_col]]) %>%
    summarise(
      Engineered_Feature = col_name,
      Positive_Count = sum(.data[[col_name]] == 1, na.rm = TRUE),
      Total = n(),
      Prevalence_Pct = round((sum(.data[[col_name]] == 1, na.rm = TRUE) / n()) * 100, 2),
      .groups = "drop"
    )
  eng_list[[col_name]] <- sub_summary
}

eng_plot_df <- bind_rows(eng_list)

cat("=== Engineered Features Prevalence Summary ===\n")
print(head(eng_plot_df, 15))
write.csv(eng_plot_df, file.path(csv_dir, "engineered_features_summary.csv"), row.names = FALSE)

# Plot: Engineered Features Prevalence across ESI Triage Levels
p_eng <- ggplot(eng_plot_df, aes(x = ESI, y = Prevalence_Pct, fill = ESI)) +
  geom_bar(stat = "identity", color = "black", alpha = 0.85) +
  geom_text(aes(label = sprintf("%.1f%%", Prevalence_Pct)), vjust = -0.2, size = 3.0) +
  facet_wrap(~ Engineered_Feature, scales = "free_y", ncol = 4) +
  scale_fill_brewer(palette = "Set1") +
  theme_minimal(base_size = 11) +
  labs(
    title = "Engineered Clinical Feature Prevalence across ESI Triage Levels",
    subtitle = "Percentage of patients meeting clinical risk thresholds (Dyspnea, Hypotension, Tachycardia, etc. from TODO.md)",
    x = "ESI Triage Level",
    y = "Prevalence (% Positive Cases)",
    fill = "ESI Level"
  ) +
  theme(legend.position = "bottom", panel.grid.minor = element_blank())

print(p_eng)
ggsave(file.path(img_dir, "engineered_features_distribution.png"), plot = p_eng, width = 14, height = 9)
cat("Saved engineered features plot to:", file.path(img_dir, "engineered_features_distribution.png"), "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Summary Report of Generated Artifacts
# ---------------------------------------------------------
cat("=== Engineered Feature Distribution Plotting Complete ===\n")
cat("Generated image plot saved to:\n")
cat("  - Engineered Features Plot: ", file.path(img_dir, "engineered_features_distribution.png"), "\n")
cat("Generated CSV summary saved to:\n")
cat("  - Engineered Features CSV:  ", file.path(csv_dir, "engineered_features_summary.csv"), "\n")